# Lithium supply chain optimisation - Energies 2024

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sear-labs/lithium-optsc-energies-2024/blob/main/notebooks/00_walkthrough.ipynb)

The model behind [Jones (2024), *Energies* 17, 2685](https://doi.org/10.3390/en17112685).

**This notebook is thin on purpose.** It imports the package and calls it; it contains no model
logic, so it cannot drift from the code that produced the paper. To read the model, read
`src/lithium_energies/model.py`.


## 1. Install

On Colab, install the package from the repository. Locally, `pip install -e ".[dev]"` once.

In [ ]:
# Colab only - skip locally
import importlib.util, subprocess, sys
if importlib.util.find_spec("lithium_energies") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/sear-labs/lithium-optsc-energies-2024.git"], check=True)

## 2. Gurobi licence

The model is far larger than the free `pip` licence allows.

A node-locked licence file cannot work in Colab - the VM is a different machine every session - so
use WLS credentials held as **Colab secrets** (key icon, left sidebar): `GRB_WLSACCESSID`,
`GRB_WLSSECRET`, `GRB_LICENSEID`.

`SecretNotFoundError` here is the expected first run for anyone who has not added them.

In [ ]:
import gurobipy as gp

try:
    from google.colab import userdata
    options = {
        "WLSACCESSID": userdata.get("GRB_WLSACCESSID"),
        "WLSSECRET":   userdata.get("GRB_WLSSECRET"),
        # userdata.get always returns a str; Gurobi needs an int here.
        "LICENSEID":   int(userdata.get("GRB_LICENSEID")),
    }
    env = gp.Env(params=options)
    print("using WLS licence from Colab secrets")
except ImportError:
    env = gp.Env()           # local machine: gurobi.lic or env vars
    print("using the local licence")
except Exception as e:
    raise SystemExit(
        "Gurobi licence not available.
"
        "In Colab, add GRB_WLSACCESSID, GRB_WLSSECRET and GRB_LICENSEID as secrets "
        "(key icon in the left sidebar) from your Gurobi WLS account.
"
        f"Original error: {e}")

## 3. Build the instance and check it against the paper

The paper states the model had **13,556 rows** and **10,706 continuous variables**. Rebuilding from
`data/raw/` should reproduce both exactly - that is the check that the data pipeline still produces
the published instance.

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "../scripts/run_all.py"],
                     capture_output=True, text=True).stdout)

## 4. Solve

Warm-started from `data/raw/warm_start.sol`, which is an **input**: the paper's run was
itself warm-started, cumulatively, to reach 28 hours of solve time.

600 s gets within ~3 units of the published 9,511,432. It will not match exactly - the model stops
on a time limit with a ~0.03% gap still open, so the published number is an incumbent, not an
optimum. See the README.

In [ ]:
print(subprocess.run([sys.executable, "../scripts/run_all.py", "--solve", "600"],
                     capture_output=True, text=True).stdout)

## 5. What to cite

Cite **the paper** for the work and **this repository** for the code. `CITATION.cff` in the repo
root carries both, and GitHub renders a "Cite this repository" button from it.